In [2]:
import pandas as pd

In [ ]:
folder_path=r"PCAdb"

In [3]:
import os
import zipfile
import json


dataframes = {}

# Get all ZIP files and sort by filename
zip_files = sorted(
    [f for f in os.listdir(folder_path) if f.endswith('.zip')],
    reverse=True  # Latest name comes first
)

if not zip_files:
    print("No ZIP files found.")
else:
    latest_zip_name = zip_files[0]
    latest_zip_path = os.path.join(folder_path, latest_zip_name)
    print(f"Loading from latest ZIP file (by name): {latest_zip_name}")

    with zipfile.ZipFile(latest_zip_path, 'r') as z:
        for file_name in z.namelist():
            if file_name.endswith('.json'):
                try:
                    with z.open(file_name) as f:
                        data = json.load(f)

                    df = pd.json_normalize(data)
                    key = os.path.splitext(os.path.basename(file_name))[0]
                    dataframes[key] = df

                except json.JSONDecodeError as e:
                    print(f"JSON decoding failed for {file_name}: {e}")
                except Exception as e:
                    print(f"Failed to load {file_name}: {e}")



Loading from latest ZIP file (by name): AutoCare_PCAdb_enUS_JSON_20260129.zip


In [4]:
df_PA=dataframes['PartAttributes']
df_PAA=dataframes['PartAttributeAssignment']
df_P=dataframes['Parts']
df_MUOM=dataframes['MetaUOMCodes']
df_MUOMA=dataframes['MetaUomCodeAssignment']
df_CM=dataframes['CodeMaster']
df_Cat=dataframes['Categories']
df_VVA=dataframes['ValidValueAssignment']
df_VV=dataframes['ValidValues']

In [5]:
df_ValidValues = (df_VV.merge(df_VVA, left_on='ValidValueID', right_on='ValidValueID', how='left')
                        .merge(df_PAA,left_on="PAPTID",right_on='PAPTID',how='outer')
                        .merge(df_PA,left_on="PAID",right_on='PAID',how='outer')
                        .merge(df_P,left_on="PartTerminologyID",right_on='PartTerminologyID',how='outer')
                        )
df_ValidValues = df_ValidValues[df_ValidValues['ValidValue'].notna()]

In [6]:
df_ValidValues

,ValidValueID,ValidValue,ValidValueAssignmentID,PAPTID,ID,PartTerminologyID,PAID,MetaID,PAName,PADescr,PartTerminologyName,PartsDescriptionID,RevDate
0,812.0,No,198320.0,205452.0,212592.0,1005.0,30.0,89.0,Mounting Hardware Included,Describes If Any Installation Hardware Is Incl...,Auxiliary Light,29385.0,2021-06-09
1,1341.0,Yes,198319.0,205452.0,212592.0,1005.0,30.0,89.0,Mounting Hardware Included,Describes If Any Installation Hardware Is Incl...,Auxiliary Light,29385.0,2021-06-09
3,812.0,No,198314.0,205448.0,212588.0,1005.0,488.0,89.0,Wiring Harness Included,Describes If Wiring Harness Is Included,Auxiliary Light,29385.0,2021-06-09
4,1341.0,Yes,198313.0,205448.0,212588.0,1005.0,488.0,89.0,Wiring Harness Included,Describes If Wiring Harness Is Included,Auxiliary Light,29385.0,2021-06-09
5,812.0,No,198316.0,205450.0,212590.0,1005.0,512.0,89.0,Switch Included,Describes If Switch Is Included,Auxiliary Light,29385.0,2021-06-09
...,...,...,...,...,...,...,...,...,...,...,...,...,...
744505,5903.0,Pillar Vented,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
744506,5957.0,m12 x 1.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
744507,5958.0,M18 x 1.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
744508,5963.0,Fan Side,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [7]:
df_ValidValues.columns

Index(['ValidValueID', 'ValidValue', 'ValidValueAssignmentID', 'PAPTID', 'ID',
       'PartTerminologyID', 'PAID', 'MetaID', 'PAName', 'PADescr',
       'PartTerminologyName', 'PartsDescriptionID', 'RevDate'],
      dtype='object')

In [8]:
df_ValidValues=df_ValidValues[['PartTerminologyName','PAName','ValidValue',]]

In [ ]:
df_ValidValues

In [10]:
df=(df_PAA.merge(df_P,left_on="PartTerminologyID",right_on='PartTerminologyID',how='outer')
.merge(df_PA,left_on="PAID",right_on='PAID',how='outer')
.merge(df_MUOMA,left_on="PAPTID",right_on='PAPTID',how='outer')
.merge(df_MUOM,left_on="MetaUomID",right_on='MetaUOMID',how='outer')
.merge(df_CM,left_on="PartTerminologyID",right_on='PartTerminologyID',how='outer')
.merge(df_Cat,left_on="CategoryID",right_on='CategoryID',how='outer'))

In [ ]:
df_cleaned=df[['CategoryID','CategoryName', 'PartTerminologyID','PartTerminologyName', 'PAID',  'PAName']].drop_duplicates()
df_cleaned

In [12]:
df_cleaned=df_cleaned[df_cleaned['CategoryName'].notna()]
df_cleaned=df_cleaned[df_cleaned['PAName'].notna()]

In [13]:
datetext=latest_zip_name.split("_")[-1].split(".")[0]

In [14]:
df_attr=df_cleaned[["PartTerminologyName", "PAName"]].drop_duplicates()

In [ ]:
df_attr.to_csv(fr"Autocare\{datetext}_Autocare_Attributes.csv", index=False, encoding='utf-8-sig')

In [ ]:
df_cleaned.to_excel(fr"\{datetext}_Autocare_PCAdb.xlsx", index=False, engine='openpyxl',sheet_name='Autocare_PCAdb')

In [ ]:
OFolder=r"Output Folder"

In [19]:
with pd.ExcelWriter(OFolder+'\\'+f'Autocare_Attribute_Details.xlsx') as writer:  # doctest: +SKIP
    df_cleaned.to_excel(writer,index=False, sheet_name='Attribute_Details')
    df_ValidValues.to_excel(writer,index=False, sheet_name='Valid_Values')